# 03 — Complex<T> e DFT

## Usage e contratti
Grafici migrati a PlotSharp tramite MathemaSharp.PlotSharp.csx: build Release di MathemaSharp e PlotSharp richieste. Runner: dotnet run --file ops/RunMigratedCoreNotebook.cs. Due SVG validati XML; layout e tooltip non equivalenti al renderer precedente, verifica visiva ancora aperta.
Dft e' forward non normalizzata, costo O(N^2), non una FFT; input vuoto restituisce array vuoto. HermitianDot coniuga il primo argomento ed e' antilineare a sinistra; richiede lunghezze uguali. VectorDot richiede lunghezze uguali, ha fallback scalare se Vector<T> non e' supportato e puo' differire dal risultato scalare per l'ordine di riduzione. Constraint: IEEE per VectorDot, IEEE + MinMax per Complex<T>.
[Reference completa](12-api-reference.ipynb), [verifiche sui sei formati](11-consolidation.ipynb).
Kernel **JupyterNet** (C#) su runtime **.NET 11**. Prerequisito: `dotnet build src/MathemaSharp.slnx -c Release`. Da terminale: `scripts/run-notebooks.ps1` (imposta il roll-forward al runtime 11); in VS Code serve `DOTNET_ROLL_FORWARD=LatestMajor` e, finché .NET 11 è RC, `DOTNET_ROLL_FORWARD_TO_PRERELEASE=1`.

`Complex<T>` (.NET 11) è generico su `T : IFloatingPointIeee754<T>, IMinMaxValue<T>`: la libreria lo usa per `Signals.Dft<T>` e per tutta la parte quantistica. Non esiste un ordinamento naturale. Angoli in radianti. DFT forward non normalizzata O(N²), non una FFT.

In [ ]:
#load "../shared/MathemaSharp.PlotSharp.csx"
using System.Numerics;
using MathemaSharp.Core;
var z = new Complex<double>(3,4);
Console.WriteLine($"{z}: modulo={z.GetMagnitude()}, fase={z.GetPhase()}");
Console.WriteLine(Complex<double>.Sqrt(-Complex<double>.One));
Console.WriteLine(Signals.HermitianDot<double>(new[] {z},new[] {z}));
Console.WriteLine(NumericAlgorithms.Dot<Complex<double>>(new[] {z},new[] {z}));

<3; 4>: modulo=5, fase=0.9272952180016122
<0; -1>
<25; 0>
<-7; 24>

In [ ]:
int n = 16;
var samples = Enumerable.Range(0,n).Select(i => new Complex<double>(Math.Cos(2*Math.PI*3*i/n),0)).ToArray();
var spectrum = Signals.Dft<double>(samples);
SpectrumChart.Show("|X[k]| — coseno a frequenza 3 su 16 campioni", spectrum.Select(c => c.GetMagnitude()).ToArray());
var leaking = Signals.Dft<double>(Enumerable.Range(0,n).Select(i => new Complex<double>(Math.Cos(2*Math.PI*3.5*i/n),0)).ToArray());
SpectrumChart.Show("|X[k]| — frequenza 3.5: leakage spettrale", leaking.Select(c => c.GetMagnitude()).ToArray());

|X[k]| — coseno a frequenza 3 su 16 campioni 0 2 4 6 8 0 5 10 15 k |X[k]| — coseno a frequenza 3 su 16 campioni · x=0: 1.2e-15 |X[k]| — coseno a frequenza 3 su 16 campioni · x=1: 1.0e-15 |X[k]| — coseno a frequenza 3 su 16 campioni · x=2: 2.2e-15 |X[k]| — coseno a frequenza 3 su 16 campioni · x=3: 8 |X[k]| — coseno a frequenza 3 su 16 campioni · x=4: 8.3e-16 |X[k]| — coseno a frequenza 3 su 16 campioni · x=5: 6.3e-16 |X[k]| — coseno a frequenza 3 su 16 campioni · x=6: 7.4e-16 |X[k]| — coseno a frequenza 3 su 16 campioni · x=7: 6.7e-16 |X[k]| — coseno a frequenza 3 su 16 campioni · x=8: 8.3e-16 |X[k]| — coseno a frequenza 3 su 16 campioni · x=9: 7.2e-16 |X[k]| — coseno a frequenza 3 su 16 campioni · x=10: 7.4e-16 |X[k]| — coseno a frequenza 3 su 16 campioni · x=11: 9.4e-16 |X[k]| — coseno a frequenza 3 su 16 campioni · x=12: 1.2e-15 |X[k]| — coseno a frequenza 3 su 16 campioni · x=13: 8 |X[k]| — coseno a frequenza 3 su 16 campioni · x=14: 2.1e-15 |X[k]| — coseno a frequenza 3 su 16 campioni · x=15: 9.5e-16

|X[k]| — frequenza 3.5: leakage spettrale 0 2 4 6 0 5 10 15 k |X[k]| — frequenza 3.5: leakage spettrale · x=0: 1 |X[k]| — frequenza 3.5: leakage spettrale · x=1: 1.129 |X[k]| — frequenza 3.5: leakage spettrale · x=2: 1.705 |X[k]| — frequenza 3.5: leakage spettrale · x=3: 5.025 |X[k]| — frequenza 3.5: leakage spettrale · x=4: 5.222 |X[k]| — frequenza 3.5: leakage spettrale · x=5: 1.886 |X[k]| — frequenza 3.5: leakage spettrale · x=6: 1.271 |X[k]| — frequenza 3.5: leakage spettrale · x=7: 1.057 |X[k]| — frequenza 3.5: leakage spettrale · x=8: 1 |X[k]| — frequenza 3.5: leakage spettrale · x=9: 1.057 |X[k]| — frequenza 3.5: leakage spettrale · x=10: 1.271 |X[k]| — frequenza 3.5: leakage spettrale · x=11: 1.886 |X[k]| — frequenza 3.5: leakage spettrale · x=12: 5.222 |X[k]| — frequenza 3.5: leakage spettrale · x=13: 5.025 |X[k]| — frequenza 3.5: leakage spettrale · x=14: 1.705 |X[k]| — frequenza 3.5: leakage spettrale · x=15: 1.129

## Risultato atteso
Picchi a k=3 e k=13, ampiezza 8. Con frequenza 3.5 l'energia si sparpaglia su tutti i bin: leakage spettrale. Prodotto hermitiano con coniugazione a sinistra: ⟨z,z⟩=25, prodotto bilineare z·z=−7+24i. La stessa `Signals.Dft<T>` funziona con float e con Decimal32/64/128: vedere notebook 05 e `--topic formats` della console.